# Telco Customer Churn - Step 4: Hyperparameter Tuning via RandomizedSearchCV

In this notebook, we fine-tune our top tree-based ensemble models (**Random Forest** and **Gradient Boosting**) using `RandomizedSearchCV` with 5-fold cross-validation, optimizing for **ROC-AUC** (`scoring='roc_auc'`) while preserving class balancing.

### Pipeline Structure:
1. **Load exact dataset & preprocessing**: `data/cleaned_churn.csv`, one-hot encoding with `drop_first=True`, stratified 80/20 split (`random_state=42`), and `StandardScaler` fitted strictly on `X_train`.
2. **Random Forest Tuning**: 5-fold stratified CV search over `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, and `max_features`.
3. **Gradient Boosting Tuning**: 5-fold stratified CV search over `n_estimators`, `learning_rate`, `max_depth`, `subsample`, `min_samples_split`, and `min_samples_leaf`.
4. **Benchmark on Unseen Test Set**: Compare Default Balanced vs. Tuned models across Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
5. **Churn Detection Analysis**: Confusion matrix evaluation on churners caught vs. missed.

In [1]:
from pathlib import Path
import time
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 1. Load Data & Prepare Identical Train/Test Splits

In [2]:
csv_path = Path("../data/cleaned_churn.csv")
if not csv_path.exists():
    csv_path = Path("data/cleaned_churn.csv")

df = pd.read_csv(csv_path)
X = df.drop(columns=["Churn"])
y = df["Churn"]

categorical_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod"
]
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dtype=int)

# Identical stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

# Scale numeric features strictly fitting on X_train
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(f"Train samples: {X_train.shape[0]:,} | Test samples: {X_test.shape[0]:,}")

Train samples: 5,634 | Test samples: 1,409


## 2. RandomizedSearchCV Setup (5-Fold Stratified CV, scoring='roc_auc')

We use `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` and sample balanced weights to ensure minority churn class prioritization throughout cross-validation.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sample_weights_train = compute_sample_weight("balanced", y_train)

## 3. Tune Random Forest Classifier

In [4]:
param_dist_rf = {
    "n_estimators": [100, 150, 200, 250],
    "max_depth": [4, 6, 8, 10, 12],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 6],
    "max_features": ["sqrt", "log2", 0.4, 0.6],
}

rf_base = RandomForestClassifier(class_weight="balanced", random_state=42)
rs_rf = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist_rf,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)

t0 = time.time()
rs_rf.fit(X_train_scaled, y_train)
print(f"Random Forest search completed in {time.time() - t0:.1f}s")
print(f"Best 5-Fold CV ROC-AUC: {rs_rf.best_score_:.4f}")
print("Best Hyperparameters:", rs_rf.best_params_)

Random Forest search completed in 28.9s
Best 5-Fold CV ROC-AUC: 0.8472
Best Hyperparameters: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 6, 'max_features': 'log2', 'max_depth': 8}


## 4. Tune Gradient Boosting Classifier

In [5]:
param_dist_gb = {
    "n_estimators": [100, 150, 200, 250],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5],
    "min_samples_split": [5, 10, 20, 30],
    "min_samples_leaf": [2, 4, 8, 12],
    "subsample": [0.7, 0.8, 0.9, 1.0],
}

gb_base = GradientBoostingClassifier(random_state=42)
rs_gb = RandomizedSearchCV(
    estimator=gb_base,
    param_distributions=param_dist_gb,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)

t0 = time.time()
rs_gb.fit(X_train_scaled, y_train, sample_weight=sample_weights_train)
print(f"Gradient Boosting search completed in {time.time() - t0:.1f}s")
print(f"Best 5-Fold CV ROC-AUC: {rs_gb.best_score_:.4f}")
print("Best Hyperparameters:", rs_gb.best_params_)

Gradient Boosting search completed in 26.0s
Best 5-Fold CV ROC-AUC: 0.8489
Best Hyperparameters: {'subsample': 0.7, 'n_estimators': 250, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 3, 'learning_rate': 0.03}


## 5. Benchmark Comparison on Unseen 20% Test Set

In [6]:
def evaluate(model, X, y):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]
    return {
        "Accuracy": accuracy_score(y, preds),
        "Precision": precision_score(y, preds),
        "Recall": recall_score(y, preds),
        "F1-Score": f1_score(y, preds),
        "ROC-AUC": roc_auc_score(y, probs),
    }

# Reference untuned balanced models
rf_untuned = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight="balanced", random_state=42)
rf_untuned.fit(X_train_scaled, y_train)

gb_untuned = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
gb_untuned.fit(X_train_scaled, y_train, sample_weight=sample_weights_train)

tuned_rf = rs_rf.best_estimator_
tuned_gb = rs_gb.best_estimator_

comparison = pd.DataFrame({
    "Random Forest (Default Bal)": evaluate(rf_untuned, X_test_scaled, y_test),
    "Random Forest (Tuned)":       evaluate(tuned_rf, X_test_scaled, y_test),
    "Gradient Boosting (Default Bal)": evaluate(gb_untuned, X_test_scaled, y_test),
    "Gradient Boosting (Tuned)":       evaluate(tuned_gb, X_test_scaled, y_test),
}).T

display(comparison.round(4))

print("\n--- Churn Detection (Test Set: 374 Total Churners) ---")
for name, m in [
    ("Random Forest (Default Bal)", rf_untuned),
    ("Random Forest (Tuned)", tuned_rf),
    ("Gradient Boosting (Default Bal)", gb_untuned),
    ("Gradient Boosting (Tuned)", tuned_gb),
]:
    cm = confusion_matrix(y_test, m.predict(X_test_scaled))
    caught, missed = cm[1, 1], cm[1, 0]
    print(f"{name:<34} -> Caught: {caught:3d} ({caught/374*100:.1f}%) | Missed: {missed:3d} ({missed/374*100:.1f}%)")

,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Random Forest (Default Bal),0.7587,0.5316,0.7647,0.6272,0.8417
Random Forest (Tuned),0.7573,0.5285,0.7941,0.6346,0.8459
Gradient Boosting (Default Bal),0.7452,0.5132,0.7781,0.6185,0.8388
Gradient Boosting (Tuned),0.7502,0.5190,0.8021,0.6303,0.8476



--- Churn Detection (Test Set: 374 Total Churners) ---
Random Forest (Default Bal)        -> Caught: 286 (76.5%) | Missed:  88 (23.5%)
Random Forest (Tuned)              -> Caught: 297 (79.4%) | Missed:  77 (20.6%)
Gradient Boosting (Default Bal)    -> Caught: 291 (77.8%) | Missed:  83 (22.2%)
Gradient Boosting (Tuned)          -> Caught: 300 (80.2%) | Missed:  74 (19.8%)


## 6. Top Predictors in Tuned Models

In [7]:
feat_importances = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Tuned RF Importance": tuned_rf.feature_importances_,
    "Tuned GB Importance": tuned_gb.feature_importances_,
})

print("Top 8 Features by Tuned Gradient Boosting Importance:")
display(feat_importances.sort_values(by="Tuned GB Importance", ascending=False).head(8).round(4))

Top 8 Features by Tuned Gradient Boosting Importance:


,Feature,Tuned RF Importance,Tuned GB Importance
18,Contract_Two year,0.1338,0.2177
1,tenure,0.1780,0.2067
9,InternetService_Fiber optic,0.1088,0.1216
17,Contract_One year,0.0524,0.1114
3,TotalCharges,0.1220,0.0783
2,MonthlyCharges,0.1006,0.0610
21,PaymentMethod_Electronic check,0.0709,0.0526
10,InternetService_No,0.0618,0.0491
